## Informationsgesellschaft und Datensicherheit

[Video1](https://youtu.be/pAjwAYoq2bY) - [Video2](https://youtu.be/yOYWkSyEue8) - [Video3](https://youtu.be/LLVNyTeWbV8)


### Sicherheitsziele

Kryptografische Verfahren dienen dazu, die folgenden Sicherheitsziele zu erreichen:

| Sicherheitsziel | Bedeutung | Wer greift an? |
|:--|:--|:--|
| **Vertraulichkeit** | Niemand außer dem Empfänger kann die Nachricht lesen. | Eve liest mit. |
| **Integrität** | Die Nachricht ist unterwegs nicht verändert worden. | Mallory verändert die Nachricht. |
| **Authentizität** | Die Nachricht stammt wirklich von dem angegebenen Absender. | Mallory gibt sich als Alice aus. |
| **Verbindlichkeit** | Der Absender kann später nicht abstreiten, die Nachricht geschickt zu haben. | Alice selbst will ihre Nachricht abstreiten. |

In diesem Kapitel lernen wir, mit welchen Methoden diese Ziele erreicht werden:

| Sicherheitsziel | erreicht durch |
|:--|:--|
| Vertraulichkeit | Verschlüsselung (symmetrisch, asymmetrisch, hybrid) |
| Integrität | Hashfunktion zusammen mit einer digitalen Signatur |
| Authentizität | digitale Signatur, Zertifikate (Public-Key-Infrastruktur) |
| Verbindlichkeit | digitale Signatur |

### Symmetrische Verschlüsselung

Bei einem **symmetrischen** Verfahren benutzen Sender und Empfänger **denselben geheimen Schlüssel** zum Ver- und Entschlüsseln. Alle Verfahren aus dem ersten Teil (Skytale, Cäsar, Vigenère, One-Time-Pad) sind symmetrisch.

Das wichtigste symmetrische Verfahren in der Praxis ist **AES** (Advanced Encryption Standard). Es wird zum Beispiel bei der Festplattenverschlüsselung, im WLAN (WPA2/WPA3) und bei HTTPS für die eigentlichen Nutzdaten eingesetzt.

Symmetrische Verfahren sind **schnell** und für große Datenmengen gut geeignet. Sie haben aber zwei Probleme:

1. **Schlüsseltausch:** Der gemeinsame Schlüssel muss vor der Kommunikation auf einem **sicheren Weg** ausgetauscht werden. Wer den Schlüssel unterwegs abfängt, kann alles mitlesen.
2. **Anzahl der Schlüssel:** Sollen sich *n* Personen paarweise vertraulich austauschen, braucht jedes Paar einen eigenen Schlüssel.

In [4]:
def anzahl_schluessel(n):
    '''Anzahl der Schlüssel, damit sich n Personen paarweise austauschen können.'''
    return n * (n - 1) // 2

for n in [2, 5, 10, 100, 1000]:
    print(f'{n:5d} Personen -> {anzahl_schluessel(n):7d} symmetrische Schlüssel')

    2 Personen ->       1 symmetrische Schlüssel
    5 Personen ->      10 symmetrische Schlüssel
   10 Personen ->      45 symmetrische Schlüssel
  100 Personen ->    4950 symmetrische Schlüssel
 1000 Personen ->  499500 symmetrische Schlüssel


Jede der *n* Personen braucht einen Schlüssel für jede der *n − 1* anderen Personen. Dabei wird jeder Schlüssel doppelt gezählt (A–B und B–A), daher $\frac{n \cdot (n-1)}{2}$.

Bei 1000 Personen sind das fast eine halbe Million Schlüssel, die alle geheim erzeugt, verteilt und verwaltet werden müssen. Die **asymmetrische Verschlüsselung** löst diese Probleme.

### Asymmetrische Verschlüsselung

Jede Person besitzt ein **Schlüsselpaar**:

- **Öffentlicher Schlüssel** (*public key*): darf frei veröffentlicht werden. Mit ihm wird **verschlüsselt**.
- **Privater Schlüssel** (*private key*): bleibt geheim. Nur mit ihm kann man **entschlüsseln**.

```
Alice möchte Bob schreiben:
  Alice verschlüsselt mit  Bobs öffentlichem Schlüssel
  Bob   entschlüsselt mit  seinem privaten Schlüssel
```

Man kann sich das wie eine Adresse mit einem **Briefkasten** vorstellen: Jeder kann an der öffentlichen bekannten Adresse einen Brief einwerfen (öffentlicher Schlüssel), aber nur der Besitzer kann den Briefkasten mit seinem Schlüssel öffnen (privater Schlüssel).

Ein Lauscher kennt zwar Bobs öffentlichen Schlüssel, kann damit aber **nicht entschlüsseln**. Der öffentliche Schlüssel darf deshalb offen über das Internet übertragen werden. **Ein geheimer Schlüsseltausch entfällt.**



### Vergleich

| Kriterium | symmetrisch (z. B. AES) | asymmetrisch (z. B. RSA) |
|---|---|---|
| **Schlüssel** | ein gemeinsamer geheimer Schlüssel | Schlüsselpaar: öffentlich + privat |
| **Schlüsselverwaltung** | bei *n* Teilnehmern bis zu *n·(n−1)/2* Schlüssel | pro Person genau ein Schlüsselpaar, also *n* Paare |
| **Schlüsseltausch** | geheimer Kanal nötig, kritisch | öffentlicher Schlüssel wird offen verteilt |
| **Geschwindigkeit** | sehr schnell, auch für große Datenmengen | rechenaufwendig, etwa 100- bis 1000-mal langsamer |
| **typischer Einsatz** | Verschlüsselung der eigentlichen Nutzdaten | Schlüsselaustausch, Signaturen |

### Hybride Verschlüsselung

In der Praxis (HTTPS, Messenger, E-Mail-Verschlüsselung) werden beide Ansätze kombiniert:

1. Es wird ein **zufälliger symmetrischer Sitzungsschlüssel** erzeugt.
2. Dieser kurze Schlüssel wird **asymmetrisch** verschlüsselt übertragen (einfacher Schlüsseltausch).
3. Die eigentliche, oft große Nachricht wird **symmetrisch** mit dem Sitzungsschlüssel verschlüsselt (hohe Geschwindigkeit).

So verbindet man den einfachen Schlüsseltausch der asymmetrischen Verfahren mit der Geschwindigkeit der symmetrischen.

### Rechnen modulo n

Asymmetrische Verfahren beruhen auf Rechnungen mit **Resten**. Bevor wir Diffie-Hellman und RSA verstehen können, lernen wir deshalb das Rechnen in **Restklassenringen** kennen.

Wir kennen es eigentlich schon von der Uhr: 5 Stunden nach 10 Uhr ist es 3 Uhr. Auf dem Zifferblatt rechnen wir **modulo 12**.

##### Rest bei der Division

Teilen wir eine ganze Zahl $a$ durch eine natürliche Zahl $n$, erhalten wir einen ganzzahligen Quotienten $q$ und einen Rest $r$:

$$a = q \cdot n + r \quad \text{mit} \quad 0 \le r < n$$

Der Rest ist nach Definition **nie negativ**.

<img src='./rest01.png' width='700'>

Bei negativen Zahlen müssen wir aufpassen: $-5 : 7 = 0 \text{ Rest } {-5}$ ist falsch, weil der Rest negativ wäre. Richtig ist $-5 = (-1) \cdot 7 + 2$, also Rest 2. Am Zahlenstrahl gehen wir von der Zahl aus in 7er-Schritten nach rechts, bis wir zwischen 0 und 6 landen:

<img src='./modulo02.png' width='701'>

In Python liefert `//` den ganzzahligen Quotienten und `%` den Rest. Auch Python gibt bei negativen Zahlen den nicht negativen Rest aus:

In [5]:
print(12 // 7, 12 % 7)
print(-5 // 7, -5 % 7)
print(-13 // 7, -13 % 7)

1 5
-1 2
-2 1


##### Der Restklassenring $\mathbb{Z}_n$

Beim Rechnen modulo 7 fassen wir alle Zahlen zusammen, die bei der Division durch 7 **denselben Rest** lassen. Sie bilden eine **Restklasse**. Zum Beispiel gehören $\dots, -5, 2, 9, 16, \dots$ zur Restklasse $\bar{2}$.

Es gibt genau 7 Restklassen: $\bar{0}, \bar{1}, \dots, \bar{6}$. Sie bilden den **Restklassenring** $\mathbb{Z}_7$. Wir stellen ihn als Kreis dar:

<img src='./modulo01.png' width='600'>

Zahlen aus derselben Restklasse heißen **kongruent**:

<img src='./modulo03a.png' width='700'>

Meist schreiben wir einfach die Zahlen $0, 1, \dots, 6$ und lassen den Querstrich weg, wenn klar ist, in welchem $\mathbb{Z}_n$ wir rechnen.

##### Addition, Subtraktion und Multiplikation

Wir rechnen wie gewohnt und nehmen am Ende den Rest:

<img src='./modulo04.png' width='801'>

Am Kreis ist das eine Bewegung:

- $3 + 6$: Wir wandern erst 3 Schritte, dann 6 Schritte im Uhrzeigersinn.
- $4 - 6$: Wir wandern 4 Schritte im Uhrzeigersinn, dann 6 Schritte gegen den Uhrzeigersinn.
- $4 \cdot 3$: Wir setzen den Bogen 4 dreimal hintereinander.

In [6]:
print((3 + 6) % 7)
print((4 - 6) % 7)
print((4 * 3) % 7)

2
5
5


##### Zwischenergebnisse verkleinern

Beim Rechnen in $\mathbb{Z}_n$ dürfen wir **jede Zahl jederzeit durch eine andere Zahl aus derselben Restklasse ersetzen**, auch durch eine negative. Das Ergebnis ändert sich dadurch nicht. So bleiben die Zahlen beim Rechnen von Hand klein.

Beispiel in $\mathbb{Z}_7$: $7050 \cdot 495 + 1208 \cdot 67$

Es ist $7050 = 1$, $495 = 5$, $1208 = 4$ und $67 = 4$ in $\mathbb{Z}_7$. Also:

$$7050 \cdot 495 + 1208 \cdot 67 = 1 \cdot 5 + 4 \cdot 4 = 21 = 0$$

Ebenso dürfen wir in $\mathbb{Z}_7$ statt $6$ auch $-1$ schreiben oder statt $4$ auch $-3$. Das wird beim Potenzieren noch nützlich.

In [7]:
print((7050 * 495 + 1208 * 67) % 7)    # große Zahlen
print((1 * 5 + 4 * 4) % 7)             # vorher verkleinert: gleiches Ergebnis

0
0


##### Division

Dividieren heißt, mit dem **Kehrwert** zu multiplizieren. Der Kehrwert $\frac{1}{5}$ in $\mathbb{Z}_7$ ist die Zahl $x$, für die $5 \cdot x = 1$ gilt:

<img src='./division01.png' width='700'>

Für kleine Zahlen finden wir den Kehrwert durch Ausprobieren:

In [8]:
for x in range(7):
    print(x, 5 * x % 7)

0 0
1 5
2 3
3 1
4 6
5 4
6 2


In der Zeile mit dem Ergebnis 1 steht $x = 3$, also ist $\frac{1}{5} = 3$ in $\mathbb{Z}_7$. Damit können wir jeden Bruch mit dem Nenner 5 berechnen:

<img src='./division02.png' width='600'>

**Nicht in jedem Restklassenring gibt es jeden Kehrwert.** In $\mathbb{Z}_8$ gibt es $\frac{1}{2}$ nicht: $2 \cdot x$ ist immer gerade, der Rest bei Division durch 8 also auch. Er kann nie 1 sein:

In [10]:
for x in range(8):
    print(x, 5 * x % 8)

0 0
1 5
2 2
3 7
4 4
5 1
6 6
7 3


$\frac{1}{5}$ gibt es in $\mathbb{Z}_8$ dagegen: $5 \cdot 5 = 25 = 1$.

Allgemein gilt:

> In $\mathbb{Z}_n$ gibt es den Kehrwert $\frac{1}{k}$ genau dann, wenn $\text{ggT}(k, n) = 1$ ist.
>
> Ist $p$ eine **Primzahl**, gilt das für alle $k \neq 0$. In $\mathbb{Z}_p$ kann man also durch jede Zahl außer 0 teilen.



##### Potenzen und diskreter Logarithmus

Potenzen mit natürlichen Exponenten berechnen wir durch wiederholtes Multiplizieren. In $\mathbb{Z}_7$ gilt zum Beispiel:

$$4^5 = 4 \cdot 4 \cdot 4 \cdot 4 \cdot 4 = 2$$

In [15]:
print(4**12 % 7)

1


Die Umkehrung ist schwieriger: Gesucht ist ein Exponent $x$ mit $4^x = 2$ in $\mathbb{Z}_7$. Ein solches $x$ heißt **diskreter Logarithmus** von 2 zur Basis 4. Für kleine Zahlen probieren wir alle Exponenten aus:

In [18]:
for x in range(1, 7):
    print(x, 4**x % 7)

1 4
2 2
3 1
4 4
5 2
6 1


$x = 2$ ist eine Lösung, denn $4^2 = 16 = 2$.

Für Zahlen mit Hunderten von Stellen kann man nicht mehr alle Exponenten ausprobieren, und es ist **kein effizientes Verfahren** bekannt, um den diskreten Logarithmus zu berechnen. Alle anderen Rechenarten (Addieren, Subtrahieren, Multiplizieren, Dividieren und Potenzieren) gehen dagegen auch mit riesigen Zahlen schnell. Auf diesem Unterschied beruht der **Diffie-Hellman-Schlüsselaustausch**.

##### Primitivwurzel

Eine Zahl $k$ in $\mathbb{Z}_p$ heißt **Primitivwurzel**, wenn man durch Potenzieren von $k$ **alle** Zahlen in $\mathbb{Z}_p$ außer der Null erreicht.

<img src='./primitivwurzel.png' width='700'>

3 ist eine Primitivwurzel in $\mathbb{Z}_7$, 2 nicht: Die Potenzen von 2 erreichen nur 1, 2 und 4.

In [19]:
def potenzen(k, p):
    '''Liste der Potenzen k^1, k^2, ..., k^(p-1) in Z_p'''
    ergebnis = []
    for x in range(1, p):
        ergebnis.append(k**x % p)
    return ergebnis

print(potenzen(3, 7))     # alle Zahlen 1 bis 6: Primitivwurzel
print(potenzen(2, 7))     # nur 1, 2, 4: keine Primitivwurzel
print(potenzen(2, 11))
print(potenzen(9, 11))

[3, 2, 6, 4, 5, 1]
[2, 4, 1, 2, 4, 1]
[2, 4, 8, 5, 10, 9, 7, 3, 6, 1]
[9, 4, 3, 5, 1, 9, 4, 3, 5, 1]


In allen Listen ist die letzte Zahl eine 1, also $k^{p-1} = 1$. Das ist kein Zufall:

<img src='./fermat.png' width='600'>

Nach $k^{p-1} = 1$ wiederholen sich die Potenzen. Deshalb hatte $4^x = 2$ in $\mathbb{Z}_7$ unendlich viele Lösungen.

##### Schnelles Potenzieren: das Powermod-Verfahren

$19^{123}$ hat 158 Stellen. Bei Exponenten mit Hunderten von Stellen wäre es unmöglich, so oft zu multiplizieren. Mit dem **Powermod-Verfahren** reichen wenige Multiplikationen:

1. Die **Basis** reduzieren: $19 = 5$ in $\mathbb{Z}_7$. Den **Exponenten** dürfen wir **nicht** modulo 7 reduzieren!
2. Den Exponenten in eine **Dualzahl** umwandeln: $123 = (1111011)_2$. Jede Ziffer 1 steht für eine Zweierpotenz, also ist $123 = 1 + 2 + 8 + 16 + 32 + 64$.
3. Durch **fortgesetztes Quadrieren** $5^1, 5^2, 5^4, 5^8, \dots, 5^{64}$ berechnen. Jede Potenz ist das Quadrat der vorigen.
4. Die benötigten Potenzen **multiplizieren**.

<img src='./powermod.png'>

Statt 122 Multiplikationen brauchen wir nur 6 Quadrierungen und 5 Multiplikationen.

In [9]:
print(pow(19, 123, 7))       # Python hat das Verfahren eingebaut

6


##### Erweiterter euklidischer Algorithmus

Für kleine Zahlen finden wir Kehrwerte durch Ausprobieren. Für große Zahlen brauchen wir den **erweiterten euklidischen Algorithmus**. Er beruht auf dem folgenden Satz:

> Sind $a$ und $b$ natürliche Zahlen, dann gibt es ganze Zahlen $x$ und $y$ mit $a \cdot x + b \cdot y = \text{ggT}(a, b)$.

Der erweiterte euklidische Algorithmus berechnet den ggT und die beiden Zahlen $x$ und $y$. Beispiel mit $a = 110$, $b = 32$:

<img src='./ggt.png' width='600'>

**Von oben nach unten:** Wir teilen $a$ durch $b$ und notieren den Quotienten $q$ und den Rest $r$. In der nächsten Zeile wird $b$ zum neuen $a$ und $r$ zum neuen $b$. Wir hören auf, wenn der Rest 0 ist. Das letzte $b$ ist der ggT.

**Von unten nach oben:** In der untersten Zeile starten wir mit $x = 0$ und $y = 1$. In jeder Zeile darüber gilt:

$$x = y_{\text{unten}}, \qquad y = x_{\text{unten}} - q \cdot y_{\text{unten}}$$

Dabei ist $q$ der Quotient aus der eigenen Zeile. Die oberste Zeile liefert das Ergebnis: $110 \cdot 7 + 32 \cdot (-24) = 2$.



##### Kehrwerte mit dem erweiterten euklidischen Algorithmus

Ist $\text{ggT}(k, n) = 1$, liefert der Algorithmus Zahlen $x$ und $y$ mit $n \cdot x + k \cdot y = 1$. In $\mathbb{Z}_n$ ist $n \cdot x = 0$, weil es ein Vielfaches von $n$ ist. Es bleibt $k \cdot y = 1$. Also ist $y$ der Kehrwert von $k$:

$$\frac{1}{k} = y \quad \text{in } \mathbb{Z}_n$$

<img src='./bruch.png' width='600'>

### Diffie-Hellman-Schlüsselaustausch

Symmetrische Verfahren wie AES brauchen einen gemeinsamen geheimen Schlüssel. Mit dem **Diffie-Hellman-Verfahren** (1976) können Alice und Bob einen solchen Schlüssel **vereinbaren**, obwohl jeder ihre Kommunikation mitlesen kann. Der Schlüssel selbst wird dabei **nie übertragen**: Beide berechnen ihn aus öffentlichen Zahlen und ihrer eigenen geheimen Zahl.

<img src='./diffie02.png' width='500'>

**Ablauf:**

1. Alice und Bob vereinbaren öffentlich eine Primzahl $p$ und eine Basis $g$, am besten eine Primitivwurzel in $\mathbb{Z}_p$.
2. Alice wählt geheim eine Zahl $a$ und berechnet $A = g^a$ in $\mathbb{Z}_p$.
   Bob wählt geheim eine Zahl $b$ und berechnet $B = g^b$ in $\mathbb{Z}_p$.
3. Alice und Bob tauschen $A$ und $B$ öffentlich aus.
4. Beide berechnen den gemeinsamen Schlüssel $K$:
   - Alice: $K = B^a = (g^b)^a = g^{ab}$
   - Bob: $K = A^b = (g^a)^b = g^{ab}$

| wer kennt was? | |
|:--|:--|
| öffentlich (auch Eve) | $p$, $g$, $A$, $B$ |
| nur Alice | $a$ |
| nur Bob | $b$ |
| Alice und Bob | $K$ |

**Beispiel** mit $p = 13$, $g = 2$, $a = 5$, $b = 8$:

<img src='./diffie01.png'>

In [10]:
p, g = 13, 2          # öffentlich
a = 5                 # geheim, nur Alice
b = 8                 # geheim, nur Bob

A = pow(g, a, p)      # Alice schickt A an Bob
B = pow(g, b, p)      # Bob schickt B an Alice
print('öffentlich:', 'p =', p, ' g =', g, ' A =', A, ' B =', B)

K_alice = pow(B, a, p)
K_bob = pow(A, b, p)
print('Alice berechnet K =', K_alice)
print('Bob berechnet   K =', K_bob)

öffentlich: p = 13  g = 2  A = 6  B = 9
Alice berechnet K = 3
Bob berechnet   K = 3


##### Warum kann Eve den Schlüssel nicht berechnen?

Eve kennt $p$, $g$, $A$ und $B$. Um $K$ zu berechnen, bräuchte sie $a$ oder $b$. Dazu müsste sie $g^x = A$ nach $x$ auflösen, also einen **diskreten Logarithmus** berechnen:

<img src='./diffie03.png' width='400'>

Bei unserem kleinen $p$ gelingt das durch Ausprobieren:

In [12]:
for x in range(1, p):
    if pow(g, x, p) == A:
        print('Eve findet a =', x)
        print('Eve berechnet K =', pow(B, x, p))
        break

Eve findet a = 5
Eve berechnet K = 3


##### Ein realistisches Beispiel

In der Praxis verwendet man **standardisierte** Werte für $p$ und $g$. Eine verbreitete Wahl ist die **2048-Bit-Primzahl** der sogenannten *MODP-Gruppe 14* aus dem Internet-Standard RFC 3526, zusammen mit der Basis $g = 2$. Diese Primzahl hat 617 Dezimalstellen.

Alice und Bob wählen ihre geheimen Zahlen $a$ und $b$ zufällig mit 256 Bit, das sind etwa 77 Dezimalstellen. Für Zufallszahlen in der Kryptografie verwendet man in Python das Modul `secrets` statt `random`: Seine Zufallszahlen sind nicht vorhersagbar.


In [14]:
import secrets

# 2048-Bit-Primzahl aus RFC 3526 (MODP-Gruppe 14), hexadezimal angegeben
p = int('''
FFFFFFFF FFFFFFFF C90FDAA2 2168C234 C4C6628B 80DC1CD1
29024E08 8A67CC74 020BBEA6 3B139B22 514A0879 8E3404DD
EF9519B3 CD3A431B 302B0A6D F25F1437 4FE1356D 6D51C245
E485B576 625E7EC6 F44C42E9 A637ED6B 0BFF5CB6 F406B7ED
EE386BFB 5A899FA5 AE9F2411 7C4B1FE6 49286651 ECE45B3D
C2007CB8 A163BF05 98DA4836 1C55D39A 69163FA8 FD24CF5F
83655D23 DCA3AD96 1C62F356 208552BB 9ED52907 7096966D
670C354E 4ABC9804 F1746C08 CA18217C 32905E46 2E36CE3B
E39E772C 180E8603 9B2783A2 EC07A28F B5C55DF0 6F4C52C9
DE2BCBF6 95581718 3995497C EA956AE5 15D22618 98FA0510
15728E5A 8AACAA68 FFFFFFFF FFFFFFFF
'''.replace(' ', '').replace('\n', ''), 16)
g = 2

print('p =', p)

a = secrets.randbits(256)        # geheim, nur Alice
b = secrets.randbits(256)        # geheim, nur Bob
print('a =', a)
print('b =', b)

A = pow(g, a, p)                 # Alice schickt A an Bob
B = pow(g, b, p)                 # Bob schickt B an Alice
print('A =', A)
print('B =', B)

K_alice = pow(B, a, p)
K_bob = pow(A, b, p)
print('K =', K_alice)
print('Alice und Bob haben denselben Schlüssel:', K_alice == K_bob)

p = 32317006071311007300338913926423828248817941241140239112842009751400741706634354222619689417363569347117901737909704191754605873209195028853758986185622153212175412514901774520270235796078236248884246189477587641105928646099411723245426622522193230540919037680524235519125679715870117001058055877651038861847280257976054903569732561526167081339361799541336476559160368317896729073178384589680639671900977202194168647225871031411336429319536193471636533209717077448227988588565369208645296636077250268955505928362751121174096972998068410554359584866583291642136218231078990999448652468262416972035911852507045361090559
a = 57890092710446168190985229417679717580458233651162728957182120896999309263856
b = 74434194442337614511159942538888805698536653041528720898339972642669712797248
A = 256881661050544890405998382549803568325097591481961510247159711413488896214885226210992718599439825976797018614649301626129535710775671599431266742733811857201312245503233734363264592541096951880658684282793504

##### Man-in-the-middle-Angriff auf Diffie-Hellman

Diffie-Hellman schützt vor Eve, die nur **mitliest**. Es schützt aber nicht vor Mallory, der die Nachrichten **abfangen und austauschen** kann.

Mallory wählt eine eigene geheime Zahl $m$. Er fängt $A$ und $B$ ab und schickt beiden stattdessen $M = g^m$. Dann vereinbart Alice den Schlüssel $g^{am}$ mit Mallory und Bob den Schlüssel $g^{bm}$ mit Mallory. Beide glauben, mit dem anderen zu sprechen. Mallory entschlüsselt jede Nachricht, liest sie, verschlüsselt sie neu und leitet sie weiter.

<img src='./middleMan.png'>

**Abhilfe:** Alice und Bob versehen $A$ und $B$ mit einer **digitalen Signatur**. Um zu verstehen, wie das funktioniert, schauen wir uns zunächst das RSA-Verfahren an.

### RSA-Verfahren

Das **RSA-Verfahren** (1977, benannt nach Rivest, Shamir und Adleman) ist das bekannteste asymmetrische Verfahren. Alice besitzt einen **öffentlichen Schlüssel** $k_e$ (e wie *encrypt*, verschlüsseln) und einen **privaten Schlüssel** $k_d$ (d wie *decrypt*, entschlüsseln). Die beiden Schlüssel heben sich gegenseitig auf:

<img src='./rsa01.png' width='901'>

**Bezeichnungen:** Wie in den Bildern ist $m$ der RSA-Modul, $n$ die Nachricht (Klartext) und $N$ der Geheimtext. 

##### Schlüssel erzeugen

Alice erzeugt ihr Schlüsselpaar so:

1. Sie wählt zwei Primzahlen $p$ und $q$ und berechnet den **RSA-Modul** $m = p \cdot q$.
2. Sie berechnet $\tilde{m} = (p-1)(q-1)$.
3. Sie wählt einen **Verschlüsselungsexponenten** $e$ mit $1 < e < \tilde{m}$ und $\text{ggT}(e, \tilde{m}) = 1$.
4. Sie berechnet den **Entschlüsselungsexponenten** $d = \frac{1}{e}$ in $\mathbb{Z}_{\tilde{m}}$ mit dem erweiterten euklidischen Algorithmus.

Die Bedingung $\text{ggT}(e, \tilde{m}) = 1$ sorgt dafür, dass es den Kehrwert $\frac{1}{e}$ in $\mathbb{Z}_{\tilde{m}}$ gibt. Das ist nötig, weil $\tilde{m}$ keine Primzahl ist.

Der **öffentliche Schlüssel** ist $(m, e)$, der **private Schlüssel** ist $(m, d)$. Die Zahlen $p$, $q$ und $\tilde{m}$ muss Alice geheim halten.

##### Ver- und Entschlüsseln

Die Nachricht ist eine Zahl $n$ mit $0 \le n < m$.

- Bob verschlüsselt mit Alices öffentlichem Schlüssel: $N = n^e$ in $\mathbb{Z}_m$
- Alice entschlüsselt mit ihrem privaten Schlüssel: $n = N^d$ in $\mathbb{Z}_m$

**Beispiel:**

<img src='./rsa02.png' width='900'>

Nebenrechnungen:

<img src='./rsa03.png' width='900'>

<img src='./rsa04.png' width='400'>

In [ ]:
# Alice erzeugt ihr Schlüsselpaar
p, q = 7, 13
m = p * q                          # RSA-Modul
m_schlange = (p - 1) * (q - 1)
e = 11
d = pow(e, -1, m_schlange)        # 1/e in Z_m~ (erweiterter euklidischer Algorithmus)
print('öffentlicher Schlüssel:', (m, e))
print('privater Schlüssel:    ', (m, d))

# Bob verschlüsselt die Nachricht n
n = 10
N = pow(n, e, m)
print('Geheimtext:', N)

# Alice entschlüsselt
print('Klartext:  ', pow(N, d, m))

### Einwegfunktionen: Warum RSA und Diffie-Hellman sicher sind

Eine **Einwegfunktion** ist eine Funktion, die in eine Richtung leicht zu berechnen ist, deren Umkehrung aber praktisch nicht in vertretbarer Zeit gelingt.

| Verfahren | leichte Richtung | schwere Richtung |
|:--|:--|:--|
| RSA | zwei Primzahlen multiplizieren: $p \cdot q = m$ | $m$ in die Primfaktoren $p$ und $q$ zerlegen (**Faktorisieren**) |
| Diffie-Hellman | potenzieren: $g^a$ in $\mathbb{Z}_p$ | aus $g^a$ den Exponenten $a$ bestimmen (**diskreter Logarithmus**) |

Bei RSA kennt der Angreifer den öffentlichen Schlüssel $(m, e)$. Könnte er $m$ in $p$ und $q$ zerlegen, könnte er $\tilde{m} = (p-1)(q-1)$ und damit auch $d = \frac{1}{e}$ in $\mathbb{Z}_{\tilde{m}}$ berechnen, genau wie Alice. Die Sicherheit von RSA hängt also davon ab, dass niemand $m$ faktorisieren kann.



Die Multiplikation $p \cdot q$ geht in Sekundenbruchteilen. Beim Faktorisieren wächst der Aufwand **exponentiell** mit der Stellenzahl.

##### Aufwand für eine Zahl mit 300 Stellen

Wenn wir die Faktoren einer Zahl mit 300 Stellen suchen, müssen wir nur Primzahlen testen, die kleiner als $\sqrt{10^{300}} = 10^{150}$ sind. Nach dem **Primzahlsatz** gibt es unterhalb einer großen Zahl $x$ ungefähr $\frac{x}{\ln(x)}$ Primzahlen.

$\ln(10^{150}) = 150 \cdot \ln(10) \approx 150 \cdot 2{,}3 \approx 345$. Also müssen wir etwa $\frac{10^{150}}{345} \approx 3 \cdot 10^{147}$ Kandidaten testen.

Annahme: Ein Computer schafft $10^{12}$ Prüfungen pro Sekunde (eine Billion). Im Dauerbetrieb sind das pro Jahr

$$10^{12} \cdot 60 \cdot 60 \cdot 24 \cdot 365 \approx 3 \cdot 10^{19} \text{ Prüfungen}.$$

Für alle Prüfungen bräuchte er $\frac{3 \cdot 10^{147}}{3 \cdot 10^{19}} = 10^{128}$ Jahre. Zum Vergleich: Das Weltall ist etwa $1{,}4 \cdot 10^{10}$ Jahre alt.

In [ ]:
import math

kandidaten = 10**150 / math.log(10**150)
pruefungen_pro_jahr = 10**12 * 60 * 60 * 24 * 365
print(f'Kandidaten:          {kandidaten:.1e}')
print(f'Prüfungen pro Jahr:  {pruefungen_pro_jahr:.1e}')
print(f'Jahre:               {kandidaten / pruefungen_pro_jahr:.1e}')

##### Wie sicher ist das wirklich?

- Es gibt deutlich schnellere Verfahren als die Probedivision. Aber auch bei ihnen wächst der Aufwand mit der Stellenzahl so stark, dass Zahlen mit über 600 Stellen (2048 Bit) heute nicht faktorisiert werden können. Für das Faktorisieren und den diskreten Logarithmus sind nur Angriffe bekannt, deren Laufzeit mit der Stellenzahl **viel stärker als polynomiell** wächst.
- **Bewiesen ist das nicht.** Niemand hat gezeigt, dass es kein schnelles Verfahren zum Faktorisieren gibt. Man kennt bisher nur keines. Es ist also denkbar, dass eines Tages doch eines gefunden wird.
- **Die Schlüssel werden mit der Zeit länger.** Computer werden immer schneller, und die bekannten Angriffsverfahren werden laufend verbessert. Was heute sicher ist, kann deshalb in einigen Jahren zu knacken sein. RSA-Schlüssel mit 512 Bit galten früher als sicher und wurden 1999 erstmals geknackt. Heute sind mindestens 2048 Bit üblich, das Bundesamt für Sicherheit in der Informationstechnik (BSI) empfiehlt für neue Anwendungen sogar mindestens 3000 Bit.
- **Quantencomputer** könnten RSA und Diffie-Hellman in Zukunft brechen. Für sie ist nämlich ein ganz anderes Verfahren bekannt, bei dem der Aufwand für das Faktorisieren und den diskreten Logarithmus mit der Stellenzahl nur noch langsam wächst. Heutige Quantencomputer sind dafür aber noch viel zu klein, und niemand weiß genau, wann es so weit sein wird. Trotzdem werden bereits neue Verfahren eingeführt, die auch gegen Quantencomputer sicher sein sollen (Post-Quanten-Kryptografie).

### Kryptographische Hashfunktionen

Eine **Hashfunktion** bildet eine beliebig lange Eingabe (einen Text, eine Datei) auf einen Wert **fester Länge** ab, den **Hashwert** der Eingabe. SHA-256 zum Beispiel liefert immer 256 Bit.

Eine **kryptographische** Hashfunktion muss zusätzlich folgende Eigenschaften haben:

- **Einwegfunktion:** Aus dem Hashwert lässt sich die Eingabe praktisch nicht rekonstruieren.
- **Kollisionsresistent:** Es ist praktisch unmöglich, zwei verschiedene Eingaben mit demselben Hashwert zu finden.
- **Lawineneffekt:** Schon eine winzige Änderung der Eingabe ergibt einen völlig anderen Hashwert.

Der Hashwert ist wie ein **Fingerabdruck** der Eingabe.

In [20]:
from hashlib import sha256

h = sha256('Dies ist eine geheime Botschaft'.encode('utf-8')).hexdigest()
print(h)
print(len(h), 'Hexadezimalziffern =', len(h) * 4, 'Bit')    # jede Hexziffer steht für 4 Bit

e61000e5186a1e65c85ad713d438f85a3b2028eb1c7b9cfe69c1954c3976146a
64 Hexadezimalziffern = 256 Bit


Wir hängen nur ein Ausrufezeichen an. Der Hashwert ist danach völlig anders:

In [21]:
print(sha256('Dies ist eine geheime Botschaft'.encode('utf-8')).hexdigest())
print(sha256('Dies ist eine geheime Botschaft!'.encode('utf-8')).hexdigest())

e61000e5186a1e65c85ad713d438f85a3b2028eb1c7b9cfe69c1954c3976146a
8f8ac630bbc84dcb4e05f499eb1d3d8cd21ff50a1383eb5bca207de91c01490f


**Anwendungen:**

- **Digitale Signatur:** Statt einer langen Nachricht wird nur ihr kurzer Hashwert signiert.
- **Fingerprint:** Mit dem Hashwert prüft man, ob eine heruntergeladene Datei unverändert ist.
- **Passwörter:** Webseiten speichern nicht das Passwort selbst, sondern nur seinen Hashwert. Beim Anmelden wird der Hashwert des eingegebenen Passworts verglichen.

### Digitale Signatur

Mit dem RSA-Verfahren und einer kryptographischen Hashfunktion kann Alice eine Nachricht **digital unterschreiben**. Dabei werden die Schlüssel umgekehrt benutzt wie beim Verschlüsseln: Alice rechnet mit ihrem **privaten** Schlüssel, jeder kann mit ihrem **öffentlichen** Schlüssel prüfen.

<img src='./signatur.png'>

Im Bild steht $N$ für die Nachricht und *sig* für die Signatur.

**Signieren (Alice):** Alice berechnet den Hashwert ihrer Nachricht und rechnet ihn mit ihrem **privaten** Schlüssel $d_A$ um: $sig = hash(N)^{d_A}$ in $\mathbb{Z}_{m_A}$. Sie verschickt die Nachricht zusammen mit der Signatur.

**Prüfen (Bob):** Bob rechnet die Signatur mit Alices **öffentlichem** Schlüssel $e_A$ zurück: $sig^{e_A}$ in $\mathbb{Z}_{m_A}$. Das ergibt den Hashwert, den Alice gebildet hat. Bob berechnet selbst den Hashwert der empfangenen Nachricht und vergleicht. Stimmen beide überein, ist die Signatur gültig.


**Welche Sicherheitsziele erreicht die Signatur?**

- **Integrität:** Wird die Nachricht unterwegs verändert, passt der neu berechnete Hashwert nicht mehr zur Signatur.
- **Authentizität:** Eine gültige Signatur kann nur erzeugen, wer Alices privaten Schlüssel besitzt.
- **Verbindlichkeit:** Aus demselben Grund kann Alice später nicht abstreiten, die Nachricht signiert zu haben.
- **Vertraulichkeit** erreicht die Signatur **nicht**: Die Nachricht selbst wird unverschlüsselt verschickt. Soll sie geheim bleiben, muss sie zusätzlich verschlüsselt werden.

Auf diese Weise schützt man auch den Diffie-Hellman-Schlüsselaustausch vor dem Man-in-the-middle-Angriff: Alice und Bob signieren ihre Werte $A$ und $B$.

Alle diese Prüfungen setzen voraus, dass Bob den **richtigen** öffentlichen Schlüssel von Alice hat.

### Public-Key-Infrastruktur (PKI)

Ein Problem asymmetrischer Verfahren ist die **Authentizität des öffentlichen Schlüssels**: Gehört der öffentliche Schlüssel wirklich zu der Person oder Webseite, die ihn präsentiert? Ein Angreifer könnte auch seinen eigenen öffentlichen Schlüssel als den von Alice ausgeben.

Deshalb lässt Alice ihren öffentlichen Schlüssel von einer anerkannten Stelle bestätigen, einem **Trust-Center** (auch Zertifizierungsstelle, englisch *Certificate Authority*, CA):

<img src='./zertifizierung.png'>

Im Bild bedeutet „verschlüsselt mit $(m_V, d_V)$": Das Trust-Center **signiert** mit seinem privaten Schlüssel.

**Ablauf:**

1. Das Trust-Center prüft die Identität der Antragstellerin, zum Beispiel mit dem Personalausweis.
2. Es erstellt ein **Zertifikat**, das den öffentlichen Schlüssel und die Daten der Inhaberin enthält.
3. Es **signiert** das Zertifikat mit seinem eigenen privaten Schlüssel.
4. Wer den öffentlichen Schlüssel des Trust-Centers kennt und ihm vertraut, kann die Signatur prüfen und sich so von der Echtheit des enthaltenen Schlüssels überzeugen.

**Woher kennt Bob den öffentlichen Schlüssel des Trust-Centers?** Browser und Betriebssysteme bringen eine Liste vertrauenswürdiger **Wurzelzertifikate** mit. Oft ist ein Zertifikat von einer Zwischenstelle signiert, deren Zertifikat wieder von einer Wurzelstelle signiert ist. So entsteht eine **Zertifikatskette** (hierarchisches Vertrauensmodell).

##### Anwendungsbereiche

| Anwendung | Wozu dient das Zertifikat? |
|:--|:--|
| **Transportverschlüsselung** (HTTPS, TLS) | Der Browser prüft, ob der Server wirklich zur aufgerufenen Adresse gehört (Schloss-Symbol). |
| **Mailverschlüsselung** (S/MIME) | E-Mails werden signiert und verschlüsselt, das Zertifikat bestätigt den Absender. |
| **Digitaler Personalausweis** (eID) | Ausweis und Behörden-Server weisen sich gegenseitig mit Zertifikaten aus. |

##### Sicherheitsaspekte

- **Das ganze System hängt am Vertrauen in die Trust-Center.** Gerät der private Schlüssel eines Trust-Centers in falsche Hände, kann der Angreifer gefälschte Zertifikate ausstellen und sich als beliebige Stelle ausgeben. Dann müssen alle Zertifikate dieses Trust-Centers zurückgezogen werden. Das ist 2011 bei der niederländischen Zertifizierungsstelle DigiNotar passiert.
- **Ein Zertifikat bestätigt nur die Identität, nicht die Seriosität.** Auch eine Phishing-Seite kann ein gültiges Zertifikat für ihre eigene (ähnlich klingende) Adresse haben.
- **Zertifikate laufen ab und können gesperrt werden.** Programme müssen das prüfen.
- **Nutzerinnen und Nutzer** klicken Zertifikatswarnungen im Browser oft einfach weg und hebeln den Schutz damit aus.

### Datensicherheit und Datenschutz

**Datensicherheit** meint den Schutz von Daten vor Verlust, Manipulation und unbefugtem Zugriff, unabhängig davon, um welche Daten es sich handelt.

**Datenschutz** meint den Schutz des Rechts der betroffenen Personen auf **informationelle Selbstbestimmung**. Es dürfen nur die wirklich nötigen personenbezogenen Daten für einen festgelegten Zweck erhoben und verarbeitet werden. Grundsätze sind **Datensparsamkeit**, **Zweckbindung** und **Einwilligung**.

##### Beispiel

Eine Schule speichert Noten in einer Datenbank.

- **Datensicherheit:** Die Datenbank wird durch Passwörter geschützt und regelmäßig gesichert.
- **Datenschutz:** Es werden nur die nötigen Daten gespeichert, nur berechtigte Lehrkräfte dürfen die Noten einsehen, und die Daten werden nicht ohne Erlaubnis weitergegeben.

##### Maßnahmen zur Gewährleistung von Datensicherheit und Datenschutz

- **technisch:** verschlüsselte Speicherung und verschlüsselte Übertragung (HTTPS), Zugriffskontrolle mit persönlichen Konten, starke Passwörter und **Mehrfaktorauthentifizierung**, regelmäßige Sicherungskopien (Backups), aktuelle Software und Sicherheitsupdates
- **organisatorisch:** Rollen- und Rechtekonzept (jede Person sieht nur die für ihre Aufgabe nötigen Daten), Schulung der Mitarbeitenden, klare Regeln zur Weitergabe und Löschung von Daten, Protokollierung der Zugriffe

### Massenhafte Datenverarbeitung

Viele Dienste (zum Beispiel kostenlose Apps, soziale Netzwerke, Webshops) erheben und speichern dauerhaft große Mengen personenbezogener Daten und werten sie zentral aus. Typische Probleme für die Betroffenen:

- Aus scheinbar harmlosen Einzeldaten (zum Beispiel Standort, Nutzungszeiten) lassen sich durch **Zusammenführen** und Auswerten genaue Profile einzelner Personen erstellen (Wohnort, Arbeitsplatz, Tagesablauf, Gesundheitszustand).
- Eine spätere **Zweckänderung** oder Weitergabe an Dritte (Werbepartner, Versicherungen, Arbeitgeber) ist für die Betroffenen kaum kontrollierbar.
- Bei einem **Datenleck** können sensible Daten dauerhaft öffentlich werden.

Beim Geschäftsmodell „kostenlose App gegen Daten" wird nicht mit Geld, sondern mit persönlichen Daten bezahlt, deren Wert und spätere Verwendung die Nutzenden nicht überblicken.

##### Ein Szenario bewerten

Diese Fragen helfen bei der Bewertung einer massenhaften Datenverarbeitung:

1. Welche Daten werden erhoben, und welche davon sind für den Zweck wirklich nötig?
2. Zu welchem Zweck werden sie verarbeitet, und wer profitiert davon?
3. Haben die Betroffenen informiert eingewilligt, und können sie die Einwilligung zurückziehen?
4. Wer hat Zugriff auf die Daten, wo werden sie gespeichert, und wie lange?
5. Was kann passieren, wenn die Daten in falsche Hände geraten oder zu einem anderen Zweck genutzt werden?
6. Gibt es eine datensparsamere Alternative?

